In [1]:
import numpy as np
import jax.numpy as jnp

In [2]:
### MDP 1: DAG with terminal states. T is an abosrbing state
P = np.array([
    [  0, .5, .5,  0,  0,  0], # A, B, C, D, E, T
    [  0,  0,  0,  0,  0,  1], 
    [  0,  0,  0, .5, .5,  0], 
    [  0,  0,  0,  0,  0,  1], 
    [  0,  0,  0,  0,  0,  1], 
    [  0,  0,  0,  0,  0,  1], 
])


P_cont = np.array([
    [  0, .5, .5,  0,  0,  0], # A, B, C, D, E, T
    [  1,  0,  0,  0,  0,  0], 
    [  0,  0,  0, .5, .5,  0], 
    [  1,  0,  0,  0,  0,  0], 
    [  1,  0,  0,  0,  0,  0], 
    [  0,  0,  0,  0,  0,  1], 
])


R_edge = np.zeros([6,6]) # R(s,s')
R_edge[2,4] = 1 # Reward for going from C to E
#R(s) = sum_{s'} P(s'|s) R(s'), sum_j P_ij R_i
R = np.sum(R_edge * P,axis=-1)

I = np.eye(len(R), len(R))
# V = np.linalg.inv(I-P) @R
V= np.zeros(len(R))
R + P @ V

def get_mu(P):
    # 1. Slice out active states [A, B, C, D, E] (just like eightrooms line 231)
    P_active = P_cont[:5, :5]
    n_active = len(P_active)

    # 2. Linear solve (P^T - I) mu = 0 with sum(mu) = 1
    A = P_active.T - np.eye(n_active)
    A[-1, :] = 1.0

    b = np.zeros(n_active)
    b[-1] = 1.0

    mu_active = np.linalg.solve(A, b)
    mu_active = np.clip(mu_active, 0.0, 1)
    mu_active = mu_active / np.sum(mu_active)

    # 3. Append 0 for terminal state T so mu has length 6
    mu = np.append(mu_active, 0.0)

    return mu
    # Output: [0.4, 0.2, 0.2, 0.1, 0.1, 0.0]
mu = get_mu(P_cont)
D = np.diag(mu)
D

array([[0.4, 0. , 0. , 0. , 0. , 0. ],
       [0. , 0.2, 0. , 0. , 0. , 0. ],
       [0. , 0. , 0.2, 0. , 0. , 0. ],
       [0. , 0. , 0. , 0.1, 0. , 0. ],
       [0. , 0. , 0. , 0. , 0.1, 0. ],
       [0. , 0. , 0. , 0. , 0. , 0. ]])

In [ ]:
# Solve with backwards induction
for i in range(10):
    V = R + P @ V

# Create a constant (wrong) estimate v= -0.1 for the sake of example
v = np.zeros(len(V)) + -0.1

e = V-v
e

In [7]:
# Way 2: (V_A - v_A) (e.g. set v_b=0)
(V[0] - v[0])

np.float64(0.35)

In [17]:
# Matrix way 2:
# Construct a terminal masked version of e
term_e = e * (1-P[:,-1])
term_e

# Smoothing A->B
# (e_A - term(e)_B)^2
print('Way 2 says (e_A-e_B)^2 is ', ((V[0] - v[0]))**2)

# generally we have a difference for all edges. 
# Can we compute all the smoothing terms at once?
# sum_{ij} μi P(ij) (ei-ej)
e_diff2 = np.subtract.outer(e, term_e) # creates an N x N matrix of ei-ej
print('Matrix way says it is ', e_diff2[0,1]**2)

Energy = np.sum(mu[:, None] * (e_diff2)**2 * P_cont)  # multiplies by P_ij
print('Energy:' ,Energy)


Way 2 says (e_A-e_B)^2 is  0.12249999999999998
Matrix way says it is  0.12249999999999998
Energy: 0.13399999999999998


In [9]:
# 1. Terminal mask
mask = 1.0 - P[:, -1]
term_e = e * mask
D_term = np.diag(mu * mask)
P_term = P_cont * mask[None, :] # Continuing P with masking!!!

# 4. Form the Laplacian matrix L
L = D + D_term - 2 * (D @ P_term)

# 5. Matrix form Energy
Energy_mat = e.T @ L @ e

print("Matrix Energy:", Energy_mat)
# Output: 0.134 (matches your sum exactly)


Matrix Energy: 0.134


In [10]:
e

array([0.35, 0.1 , 0.6 , 0.1 , 0.1 , 0.1 ])

In [11]:
Energy_mat = (e.T @ D @ e) - 2 * (e.T @ D @ P_term @ e) + (term_e.T @ D @ term_e)
Energy_mat


np.float64(0.134)

In [14]:
# 1. Terminal state T has value 0 and error 0
v = np.array([-0.1, -0.1, -0.1, -0.1, -0.1, 0.0])  # v_T is fixed at 0.0
e = V - v                                          # e_T is strictly 0.0

# 2. Stationary distribution mu (from P_cont, with mu_T = 0.0)
mu = np.array([0.4, 0.2, 0.2, 0.1, 0.1, 0.0])

# -------------------------------------------------------------
# Approach A: Edge-by-Edge Sum Form (using episodic P)
# -------------------------------------------------------------
e_diff = np.subtract.outer(e, e)
Energy_sum = np.sum(mu[:, None] * (e_diff**2) * P)
print("Energy (Sum form):", Energy_sum)  # 0.079

# -------------------------------------------------------------
# Approach B: Matrix Form E = e^T @ A @ e (matches ppo/exact_E.py)
# -------------------------------------------------------------
D = np.diag(mu)
A = D @ (np.eye(6) - P)
E = e.T @ A @ e
print("E (Matrix form):", E)              # 0.064


Energy (Sum form): 0.079
E (Matrix form): 0.064


In [15]:
# Exact matrix form for Approach A:
D = np.diag(mu)
D_in = np.diag(P.T @ mu)
L = D + D_in - (D @ P + P.T @ D)

Energy_mat = e.T @ L @ e
print("Energy_mat:", Energy_mat)  # Exactly 0.079!


Energy_mat: 0.07899999999999999
